In [ ]:
from utilities import initialize_with_module

await initialize_with_module('AITW-aiida-wood-permeability/0.3.0-foss-2025b', '2025.06')

In [ ]:
# Create/load profile/computer/codes
import utilities_aiida as ua

from common import PERMEABILITY_CODES


print("Ensure required computer/codes are set up...")

profile = ua.setup_profile()
computer = ua.setup_computer()
codes = ua.setup_codes(code_infos=PERMEABILITY_CODES)

print("Using profile:")
print(f" - {profile}")
print("Using computer:")
print(f" - {computer}")
print("Using codes: ")
for code in codes:
    print(f" - {code}")

# create_codes_button.on_click(on_create_codes_button_clicked)
# display(create_codes_button, output_area)

In [ ]:
from aiida_wood_permeability.cli.workflows.full import launch_workflow

from common import PERMEABILITY_WORKCHAIN_DEFAULTS, PERMEABILITY_CODE_MAP
import utilities

widgets_launch, launch_event = utilities.click_cli_widget(
    launch_workflow,
    title="AITW - AiiDA OpenLB Permeability Workchain",
    override_defaults=PERMEABILITY_WORKCHAIN_DEFAULTS,
    code_map=PERMEABILITY_CODE_MAP,
    hide_params=['daemon'],
)
display(widgets_launch)

In [ ]:
import json

from aiida import orm
from aiida.plugins import WorkflowFactory

import utilities
from utilities_aiida import with_aiida_profile

def get_outputs() -> list[tuple[int, str]]:
    """Get the outputs of the completed workchaisn."""
    workchain = WorkflowFactory('aitw.wood_permeability.wood_permeability')
    # choices = []
    with with_aiida_profile():
        qb = orm.QueryBuilder()
        qb.append(workchain, tag='node', filters={'attributes.process_state': 'finished'})
        return [(node.pk, str(node.pk)) for node, in qb.all()]

def read_func(pk_str: str) -> orm.Node:
    """Read the output data for a given workchain pk."""
    with with_aiida_profile():
        return orm.load_node(int(pk_str))

def label_func(pk: int, wc: orm.Node) -> str:
    """Generate a label for the output data based on the workchain pk."""
    with with_aiida_profile():
        perm_params = wc.inputs.permeability.parameters.get_dict()
        wood_type = wc.inputs.generator.wood_type.value
        resolution = perm_params['resolution']
        scaling_factor = perm_params['scaling_factor']
        return f"{wood_type} - res={resolution} - scale={scaling_factor} (pk={pk})"

def data_display_func(wc: orm.Node):
    """Display the output data for a given workchain."""
    with with_aiida_profile():
        # Display the permeability tensor
        output_tensor = wc.outputs.permeability__permeability_tensor
        print("Permeability Tensor:")
        print(json.dumps(output_tensor.get_dict(), indent=4))


visualizer = utilities.output_visualizer_widget(
    title="Permeability Tensor Results",
    event_observer=launch_event,
    outputs_func=get_outputs,
    read_func=read_func,
    selector_label_func=label_func,
    data_display_func=data_display_func
)
display(visualizer)